In [1]:
import numpy as np 
import matplotlib.pyplot as plt 
import itertools 

In [9]:
#This code simulates with the system, with a singular freshwater impulse at time t=15.7.
#Salinity is decreased by a prescribed amount instantaneously to mimic a freshwwater impulse.
#The salinity immediately before the impulse is printed, to make sure there is only one impulse 
#when the impulse is released the word 'HAPPENED' is printed, and the time of impulse.

def Euler_step(T_1, T_2, S_1, S_2, t, dt, params, impulse):
    T_1_star, S_1_star, T_2_star, S_2_star, tau_1_T, tau_1_S, tau_2, A_T, A_S, psi=params
    #First integration
    dT1dt = (1/tau_1_T) * (T_1_star - A_T*np.cos(2*np.pi*t) -T_1)
    dS1dt = (1/tau_1_S) * (S_1_star + A_S*np.cos(2*np.pi*t + psi) - S_1)
    dT2dt = (1/tau_2) * (T_2_star - T_2)
    dS2dt = (1/tau_2) * (S_2_star - S_2)

    T_1_new = T_1 + dT1dt * dt 
    S_1_new = S_1 + dS1dt * dt 
    T_2_new = T_2 + dT2dt * dt 
    S_2_new = S_2 + dS2dt *dt 

    #Convective adjustment if statically unstable 
    
    Delta_rho = -alpha * (T_1-T_2) + beta * (S_1 - S_2)
    if Delta_rho > 0 :
        T_1_new = h_star*T_1_new + (1-h_star)*T_2_new 
        T_2_new = T_1_new 
        S_1_new = h_star*S_1_new + (1-h_star)*S_2_new
        S_2_new = S_1_new 

    else:
        T_1_new, T_2_new, S_1_new, S_2_new = T_1_new, T_2_new, S_1_new, S_2_new
        
    #Add freshwater to disrupt convection 
    if 15.7 <t < 15.701:
        print(S_1)
        S_1_new = S_1_new - impulse
        print('HAPPENED')
        print(t)
    
    return T_1_new, T_2_new, S_1_new, S_2_new
    
#Run the time integration 
def run_euler(T_1, T_2, S_1, S_2, T_max, dt, params, impulse):
    T_1_list = []
    T_2_list = []
    S_1_list = []
    S_2_list = [] 
    tlist = []
    t = 0
    #Run simulation 
    while t <= T_max:
        T_1, T_2, S_1, S_2 = Euler_step(T_1, T_2, S_1, S_2, t, dt, params, impulse)
        tlist.append(t)
        T_1_list.append(T_1)
        T_2_list.append(T_2)
        S_1_list.append(S_1)
        S_2_list.append(S_2)
        #print(t/dt)
        t += dt 
    
    return tlist, T_1_list, T_2_list, S_1_list, S_2_list 

def simulate_model(T_max, dt, params, impulse):
    tlist, T_1sim, T_2sim, S_1sim, S_2sim = run_euler(T_1init, T_2init, S_1init, S_2init, T_max, dt, params, impulse)
    return tlist[0::30], T_1sim[0::30], T_2sim[0::30], S_1sim[0::30], S_2sim[0::30]

In [10]:
#Parameters and variables for simulation 
h_star = 1/10

#Timestep 
dt = 1/360

#Initial conditions 200m box 
T_1init = 3.55
T_2init = 3.61
S_1init = 34.78
S_2init = 34.87


#Expansion coefficients 
alpha = 1
beta = 10

params = [3.84, 34.63, 4.00, 34.88, 0.32, 6.43, 1.41, 2.78, 2.99, 0.14] 



In [11]:
#This calculates the size of freshwater impulse required to cause the temporary shutdown. 
#The size of impulse is gradually increased. If a shutdown has been caused, the impulse is printed. 
#We note the first impulse to be printed as the impulse required to cause a shutdown. 

impulse = 0.1
impulse_list = [0.1 + 0.002*i for i in range(25) ]
indicator = []
for impulse in impulse_list:
    y = simulate_model(200, dt, params, impulse)
    if np.isclose(y[2][-1] - y[2][-2], 0, atol=1e-5) == True:
        print(impulse)

        
  

34.716704262001606
HAPPENED
15.700000000001815
34.716704262001606
HAPPENED
15.700000000001815
34.716704262001606
HAPPENED
15.700000000001815
0.10400000000000001
34.716704262001606
HAPPENED
15.700000000001815
0.10600000000000001
34.716704262001606
HAPPENED
15.700000000001815
0.10800000000000001
34.716704262001606
HAPPENED
15.700000000001815
0.11
34.716704262001606
HAPPENED
15.700000000001815
0.112
34.716704262001606
HAPPENED
15.700000000001815
0.114
34.716704262001606
HAPPENED
15.700000000001815
0.116
34.716704262001606
HAPPENED
15.700000000001815
0.11800000000000001
34.716704262001606
HAPPENED
15.700000000001815
0.12000000000000001
34.716704262001606
HAPPENED
15.700000000001815
0.122
34.716704262001606
HAPPENED
15.700000000001815
0.124
34.716704262001606
HAPPENED
15.700000000001815
0.126
34.716704262001606
HAPPENED
15.700000000001815
0.128
34.716704262001606
HAPPENED
15.700000000001815
0.13
34.716704262001606
HAPPENED
15.700000000001815
0.132
34.716704262001606
HAPPENED
15.700000000001

In [12]:
#Now we calculate the mass required via dS = S1 - S2 = M/hA - M/(hA + V) => V = -M/(dS - M/hA) - hA
#Where M is the total mass of salt (constant across freshwater impulse), h is the height of the column, A the area
#And V the volume of freshwater from which we calculate the mass. 
dS = 0.104 * 1025/1000
hA = 0.22271 * 1262000 * 10**9
S1 = 34.717 * 1.025
M = S1 * hA
V = -M/(dS - M/hA) - hA
V
mass_required = V * 1025 / 1e12
print('Mass required = ' + str(mass_required) + ' GT')

Mass required = 865.5995762285945 GT
